# 03_Webcam_Test.ipynb
Test webcam cho 3 model TCN, GRU và Transformer. Mỗi model có một cell chạy riêng ở cuối notebook.

**Lưu ý:** chạy các cell chuẩn bị từ trên xuống trước; sau đó chỉ chạy cell của model muốn thử. Nhấn `q` trong cửa sổ webcam để thoát.

In [1]:
#cell 1: import thư viện và kiểm tra CUDA
import cv2
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

assert torch.cuda.is_available(), "CUDA chưa bật trong môi trường train-gpu"
device = torch.device("cuda")
print("CUDA:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| MediaPipe:", mp.__version__, "| OpenCV:", cv2.__version__)

CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU
PyTorch: 2.6.0+cu124 | MediaPipe: 1.1.0 | OpenCV: 5.0.0


In [2]:
#cell 2: khai báo đường dẫn, đọc độ dài tối đa đã dùng khi train và kiểm tra file cần thiết
BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
TASK_FILE = BASE / "hand_landmarker.task"
CKPT_DIR = BASE / "checkpoints"
MANIFEST = BASE / "manifest.csv"

assert TASK_FILE.exists(), f"Không tìm thấy: {TASK_FILE}"
assert MANIFEST.exists(), f"Không tìm thấy: {MANIFEST}"
assert CKPT_DIR.exists(), f"Không tìm thấy thư mục checkpoint: {CKPT_DIR}"

import pandas as pd
manifest = pd.read_csv(MANIFEST)
T_MAX = int(manifest["khung"].max())
print("T_MAX:", T_MAX)
print("Checkpoint hiện có:", [p.name for p in CKPT_DIR.glob("*.pt")])

T_MAX: 355
Checkpoint hiện có: ['GRU.pt', 'TCN.pt', 'Transformer.pt']


In [3]:
#cell 3: định nghĩa đúng kiến trúc TCN, GRU và Transformer từ 02_Train.ipynb
class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__()
        self.pad = 2 * d
        self.conv = nn.Conv1d(c, c, 3, dilation=d)
        self.drop = nn.Dropout(0.1)

    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        self.inp = nn.Linear(126, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        self.out = nn.Linear(c, 3)

    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

class GRU(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        self.inp = nn.Linear(126, c)
        self.rnn = nn.GRU(c, c, batch_first=True)
        self.out = nn.Linear(c, 3)

    def forward(self, x):
        return self.out(self.rnn(F.relu(self.inp(x)))[0])

class Trans(nn.Module):
    def __init__(self, c=64, T=T_MAX):
        super().__init__()
        self.inp = nn.Linear(126, c)
        self.pos = nn.Parameter(torch.randn(1, T, c) * 0.02)
        self.enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(c, 4, 128, 0.1, batch_first=True), 2
        )
        self.out = nn.Linear(c, 3)

    def forward(self, x):
        t = x.shape[1]
        mask = nn.Transformer.generate_square_subsequent_mask(t, device=x.device)
        return self.out(self.enc(self.inp(x) + self.pos[:, :t], mask=mask))

LABELS = ["Paper", "Rock", "Scissors"]
print("Đã định nghĩa 3 kiến trúc model.")

Đã định nghĩa 3 kiến trúc model.


In [4]:
#cell 4: chuẩn hóa landmark giống lúc train, nội suy webcam gần 120 fps, vẽ khung xương và chạy webcam dùng chung
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(0,17),(17,18),(18,19),(19,20)
]

def normalized_position(points):
    lm = np.asarray(points, dtype=np.float32)
    lm = lm - lm[:1]                         # trừ tọa độ cổ tay
    scale = np.linalg.norm(lm[9])             # khoảng cách cổ tay -> gốc ngón giữa
    if not np.isfinite(scale) or scale < 1e-6:
        scale = 1.0
    return (lm / scale).reshape(63)

def make_feature(pos, prev_pos):
    # Vận tốc đúng như lúc train: hiệu vị trí đã chuẩn hóa giữa 2 khung liên tiếp
    vel = np.zeros(63, dtype=np.float32) if prev_pos is None else pos - prev_pos
    return np.nan_to_num(np.concatenate([pos, vel])).astype(np.float32)

def draw_hand(frame, landmarks):
    h, w = frame.shape[:2]
    pts = [(int(p.x * w), int(p.y * h)) for p in landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (70, 220, 70), 2, cv2.LINE_AA)
    for x, y in pts:
        cv2.circle(frame, (x, y), 3, (40, 120, 255), -1, cv2.LINE_AA)

def run_webcam(model_name, model_class):
    ckpt_path = CKPT_DIR / f"{model_name}.pt"
    assert ckpt_path.exists(), f"Không tìm thấy checkpoint: {ckpt_path}"

    model = model_class().to(device).eval()
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["w"])
    print(f"Đã nạp {model_name}: {ckpt_path}")

    options = vision.HandLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(TASK_FILE)),
        running_mode=vision.RunningMode.VIDEO,
        num_hands=1
    )
    detector = vision.HandLandmarker.create_from_options(options)
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        detector.close()
        raise RuntimeError("Không mở được webcam. Hãy kiểm tra camera hoặc đổi VideoCapture(0).")

    history = []
    prev_pos = None
    prev_time_ms = None
    frame_id = 0
    last_time = time.perf_counter()
    fps = 0.0
    pred_text = "Đang tìm bàn tay..."
    conf_value = 0.0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("Không đọc được khung hình webcam.")
                break

            frame = cv2.flip(frame, 1)        # lật ngang như gương
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            # Timestamp tăng đều, MediaPipe Tasks yêu cầu timestamp tăng dần
            timestamp_ms = frame_id * 33
            result = detector.detect_for_video(image, timestamp_ms)
            frame_id += 1

            if result.hand_landmarks:
                points = result.hand_landmarks[0]
                draw_hand(frame, points)
                pos = normalized_position([[p.x, p.y, p.z] for p in points])

                # Dataset train ở 120 FPS; webcam thường khoảng 30 FPS.
                # Nội suy các vị trí giữa hai lần đọc webcam để xấp xỉ 120 FPS.
                if prev_pos is None:
                    history.append(make_feature(pos, None))
                else:
                    steps = 4
                    for j in range(1, steps + 1):
                        p = prev_pos + (pos - prev_pos) * (j / steps)
                        history.append(make_feature(p, prev_pos if j == 1 else prev_interp))
                        prev_interp = p
                prev_pos = pos
                while len(history) > T_MAX:
                    history.pop(0)

                x = torch.from_numpy(np.asarray(history, dtype=np.float32)).unsqueeze(0).to(device)
                with torch.inference_mode():
                    logits = model(x)
                    probs = torch.softmax(logits[0, -1], dim=-1)
                    conf, pred = probs.max(dim=-1)
                pred_text = LABELS[int(pred.item())]
                conf_value = float(conf.item())
            else:
                # Giữ đúng nhịp thời gian: thêm khung 0 khi mất tay như lúc chuẩn bị dữ liệu
                history.extend([np.zeros(126, dtype=np.float32)] * 4)
                history = history[-T_MAX:]
                prev_pos = None
                pred_text = "Khong thay tay"
                conf_value = 0.0

            now = time.perf_counter()
            dt = now - last_time
            if dt > 0:
                fps = 0.9 * fps + 0.1 * (1.0 / dt) if fps else 1.0 / dt
            last_time = now

            cv2.putText(frame, f"Model: {model_name}", (15, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 220, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Predict: {pred_text}", (15, 62),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Confidence: {conf_value:.2f} (threshold 0.50)", (15, 94),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"FPS: {fps:.1f}", (15, 125),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 255), 2, cv2.LINE_AA)
            cv2.imshow(f"Webcam - {model_name} (press q to quit)", frame)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        detector.close()
        cv2.destroyAllWindows()
        print(f"Đã dừng webcam {model_name}.")

In [16]:
#cell 5: chạy thử model TCN (chạy cell này khi muốn test TCN)
run_webcam("TCN", TCN)

Đã nạp TCN: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints\TCN.pt
Đã dừng webcam TCN.


In [13]:
#cell 6: chạy thử model GRU (chạy cell này khi muốn test GRU)
run_webcam("GRU", GRU)

Đã nạp GRU: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints\GRU.pt
Đã dừng webcam GRU.


In [15]:
#cell 7: chạy thử model Transformer (chạy cell này khi muốn test Transformer)
run_webcam("Transformer", Trans)

Đã nạp Transformer: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints\Transformer.pt
Đã dừng webcam Transformer.
